In [1]:
# Importar funciones y módulos
import logging
import os
import pandas as pd
import seaborn as sns
from src.clustering.kmeans_modelado import (
    plot_elbow_method,
    entrenar_kmeans,
    asignar_clusters,
    resumen_clusters
)

# Configurar logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Cargar los datos procesados
ruta_data = "../data/processed/vehiculos_limpios.csv"
df_modelo = pd.read_csv(ruta_data)

# Crear carpeta de salida para gráficos si no existe
os.makedirs("../outputs/03_modelado", exist_ok=True)

In [2]:
# 1. Aplicar método del codo y guardar gráfico
plot_elbow_method(
    df=df_modelo,
    max_k=30,
    save_path="../outputs/03_modelado/elbow_method.png"
)

2025-07-02 13:05:01,143 - INFO - K=1, Inercia=290704.75
2025-07-02 13:05:01,208 - INFO - K=2, Inercia=193665.84
2025-07-02 13:05:01,254 - INFO - K=3, Inercia=154801.58
2025-07-02 13:05:01,302 - INFO - K=4, Inercia=136318.48
2025-07-02 13:05:01,359 - INFO - K=5, Inercia=123361.00
2025-07-02 13:05:01,451 - INFO - K=6, Inercia=114580.40
2025-07-02 13:05:01,553 - INFO - K=7, Inercia=110601.55
2025-07-02 13:05:01,673 - INFO - K=8, Inercia=100933.88
2025-07-02 13:05:01,755 - INFO - K=9, Inercia=98363.25
2025-07-02 13:05:01,861 - INFO - K=10, Inercia=90440.04
2025-07-02 13:05:02,007 - INFO - K=11, Inercia=86683.59
2025-07-02 13:05:02,156 - INFO - K=12, Inercia=86217.03
2025-07-02 13:05:02,268 - INFO - K=13, Inercia=81636.03
2025-07-02 13:05:02,386 - INFO - K=14, Inercia=79343.70
2025-07-02 13:05:02,500 - INFO - K=15, Inercia=78256.73
2025-07-02 13:05:02,630 - INFO - K=16, Inercia=76332.23
2025-07-02 13:05:02,753 - INFO - K=17, Inercia=73645.35
2025-07-02 13:05:02,888 - INFO - K=18, Inercia=72

Se observa un punto de inflexión para k=10 por el Método del Codo, entonces se selecciona dicho parámetro para la formación de clusters.

In [3]:
# 2. Entrenar el modelo
modelo_kmeans = entrenar_kmeans(df_modelo, k=10)

2025-07-02 13:05:06,073 - INFO - K-means entrenado con K=10


In [4]:
# 3. Asignar clusters
df_clusterizado = asignar_clusters(df_modelo, modelo_kmeans)

2025-07-02 13:05:06,136 - INFO - Clusters asignados al DataFrame


In [5]:
# 4. Analizar clusters (resumen + visualizaciones)
resumen = resumen_clusters(
    df=df_clusterizado,
    save_dir="../outputs/03_modelado",
    var_x="cilindros",
    var_y="consumo_litros_milla"
)

2025-07-02 13:05:08,348 - INFO - Resumen estadístico por cluster generado


In [6]:
# Se exporta el dataframe clusterizado

# Crear la carpeta si no existe
ruta_export = "../data/processed"
os.makedirs(ruta_export, exist_ok=True)

# Exportar archivo
ruta_archivo = os.path.join(ruta_export, "vehiculos_clusterizado.csv")
df_clusterizado.to_csv(ruta_archivo, index=False)
logging.info(f"Data con clusters exportada a: {ruta_archivo}")


2025-07-02 13:05:08,882 - INFO - Data con clusters exportada a: ../data/processed\vehiculos_clusterizado.csv


Se observa que los clusters definidos están bien balanceados, con una proporción entre el 17% y el 6% de observaciones, por lo que no hay conglomerados que únicamente atrapen observaciones ruidosas. La selección del número de clusters es coherente para un análisis comparativo.

Existe una relación positiva clara entre el número de cilindros y el consumo de combustible (en litros por milla):

- Mayor número de cilindros → mayor consumo.
- Menor número de cilindros → menor consumo.

 Desde el punto de vista mecánico: motores con más cilindros suelen ser más potentes, pero también menos eficientes en cuanto al consumo de combustible.

1. Clusters ubicados en la esquina superior derecha (Clusters 2 y 3):
- Cilindros altos y consumo alto.
- Representan vehículos potentes y menos eficientes.

2. Clusters en la parte inferior izquierda (Clusters 1 y 9):
- Cilindros bajos y consumo bajo.
- Vehículos más eficientes, probablemente económicos o compactos.

3. Clusters intermedios:
- Mezcla de configuraciones, posibles vehículos balanceados en potencia y eficiencia.